# FedTFT Preprocessing Pipeline
## Finalized pipeline 
**Stage 1 (Cell 1):** Sequence creation — loads raw CSV, runs augmentation, cosinor analysis, patient-level split, saves pickles to `processed_sequences_by_hospital/`  
**Stage 2 (Cell 2):** Memmap creation — converts pickles to memory-mapped .npy arrays in `last_npy_data/`  
  
### Config knobs for sensitivity variants:
- Cosinor windows: change `48` and `168` in `PipelineSensor()`  
- Sequence length: change `SEQ_LEN`  
- Aug sigma: change `DataAugmentor(sigma=...)`  
- w/o augmentation: comment out `train_aug = augmentor.augment_patient_data(...)`  


In [ ]:
import os
import math
import pickle
import warnings

import numpy as np
import pandas as pd
from numpy.linalg import lstsq
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit
from tqdm import tqdm

# --- CONFIG ---
RAW_CSV      = "path omitted for privacy of raw_data"
OUT_DIR      = "patient_level_split/processed_sequences_by_hospital"
CHUNK_SIZE   = 30000
SEQ_LEN      = 192
MAX_SHIFT    = 672
PERIODS      = {'1h':4, '1d':96, '1w':672}

os.makedirs(OUT_DIR, exist_ok=True)

# --- 1. LOAD & CLEAN ---
data = pd.read_csv(RAW_CSV)

STRUCTURAL     = ['이름', 'targetTime', 'idxInfo', 'dangerous_action']
LOCATION_FEATS = [
    'Daily_Entropy', 'Norm_Daily_Entropy', 'Location_Variability',
    'place_hallway', 'place_other', 'place_ward',
]
EMR_FEATS = [
    'DIG_3class', 'DIG_4class', 'DIG_withPsychosis', 'Restrictive_Interv',
    'age', 'sex', 'day_of_week', 'holidays',
]
SENSOR_BASE_FEATS = [
    'ENMO_max', 'ENMO_mean', 'ENMO_median', 'ENMO_min', 'ENMO_nunique', 'ENMO_std',
    'HR_max', 'HR_mean', 'HR_median', 'HR_min', 'HR_nunique', 'HR_std',
    'CALORIES_delta', 'DISTANCE_delta', 'SLEEP_delta', 'STEP_delta',
    'VE_tx', 'VE_tx_inj_time', 'nonwearing',
]
KEEP = STRUCTURAL + LOCATION_FEATS + EMR_FEATS + SENSOR_BASE_FEATS
data = data[[c for c in KEEP if c in data.columns]].copy()

sensor_cols = [c for c in SENSOR_BASE_FEATS if c in data.columns]
data.loc[data['nonwearing'].astype(bool), sensor_cols] = 0
data = data.fillna(0)

# rename "이름" → "key" and extract "hospital"
data.rename(columns={'이름':'key'}, inplace=True)
data['hospital'] = data['idxInfo'].str.split(pat='-', n=1).str[0]

hospitals = data['hospital'].unique()

# identify categorical vs. continuous
EXCL = ['key','targetTime','idxInfo','hospital']
TH   = 20
categorical_columns = [
    c for c in data.columns
    if c not in EXCL and (data[c].nunique() <= TH or data[c].dtype == 'object')
]
continuous_columns = [
    c for c in data.columns
    if c not in EXCL and c not in categorical_columns
    and data[c].dtype in ['int64','float64']
]

# --- 2. WINSORIZE CONTINUOUS FEATURES (1.5× IQR, Tukey's mild-outlier fence) ---
for c in continuous_columns:
    q1 = data[c].quantile(0.25)
    q3 = data[c].quantile(0.75)
    iqr = q3 - q1
    data[c] = data[c].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
print(f"Winsorized {len(continuous_columns)} continuous features (1.5× IQR bounds).")

# --- 3. HELPERS: AUGMENT, COSINOR, SPLIT, SEQ GENERATION ---

class DataAugmentor:
    def __init__(self, sigma=0.5, num_augments=5):
        self.sigma = sigma
        self.num_augments = num_augments

    def time_warping(self, data):
        warping_factor = np.random.normal(1, self.sigma, size=data.shape)
        return data * warping_factor

    def augment_patient_data(self, df, target_columns, categorical_columns):
        positive_data = df[df['dangerous_action'] == 1]
        augmented_data = []
        patient_id = max(
            [int(k.split('_')[-1]) for k in df['key'].unique() if '_' in k] + [0]
        ) + 1

        for name, group in positive_data.groupby("key"):
            for _ in range(self.num_augments):
                new_patient_data = group.copy()
                new_patient_data["key"] = f"NewPatient_{patient_id}"
                patient_id += 1

                # time-warp each target column
                for col in target_columns:
                    new_patient_data[col] = self.time_warping(new_patient_data[col].to_numpy())

                # add small noise to other numeric features
                for col in group.columns:
                    if col in categorical_columns:
                        continue
                    if pd.api.types.is_numeric_dtype(group[col]) and col not in target_columns:
                        new_patient_data[col] += np.random.normal(0, self.sigma, size=new_patient_data[col].shape)

                augmented_data.append(new_patient_data)

        if len(augmented_data) == 0:
            return df
        augmented_df = pd.concat(augmented_data, ignore_index=True)
        return pd.concat([df, augmented_df], ignore_index=True)


class CosinorAnalyzer:
    def __init__(self, data: pd.DataFrame, window_length_hours: float,
                 coverage_threshold: float = 0.5, sensor: str = 'ENMO',
                 name_col: str = '이름', time_col: str = 'targetTime'):
        self.data = data.copy()
        self.window_length_hours = window_length_hours
        self.coverage_threshold = coverage_threshold
        self.period = 24.0
        self.sensor = sensor
        self.name_col = name_col
        self.time_col = time_col
        self.sensor_col = f"{self.sensor}_mean"

        required_cols = [self.time_col, self.sensor_col, 'nonwearing', self.name_col]
        for col in required_cols:
            if col not in self.data.columns:
                raise ValueError(f"Missing column '{col}' for CosinorAnalyzer.")
        if not pd.api.types.is_datetime64_any_dtype(self.data[self.time_col]):
            raise ValueError(f"'{self.time_col}' must be datetime.")

        self.data.sort_values([self.name_col, self.time_col], inplace=True)
        self.data.reset_index(drop=True, inplace=True)

        time_diffs = self.data.groupby(self.name_col)[self.time_col].diff().dropna()
        if len(time_diffs) == 0:
            raise ValueError("Not enough data to compute sampling rate.")
        self.sampling_interval_minutes = time_diffs.median().total_seconds() / 60.0

    def _prepare_window_data(self, df_group, ref_time: pd.Timestamp, global_reference_time):
        start_time = ref_time - pd.Timedelta(hours=self.window_length_hours)
        df_window = df_group[(df_group[self.time_col] >= start_time) &
                             (df_group[self.time_col] <= ref_time)]
        df_window = df_window[df_window['nonwearing'] == False]
        expected = int((self.window_length_hours * 60) / self.sampling_interval_minutes)
        if len(df_window) == 0 or (len(df_window) / expected) < self.coverage_threshold:
            return None

        dfw = df_window.copy()
        dfw['t_hours'] = (dfw[self.time_col] - global_reference_time).dt.total_seconds() / 3600.0
        return dfw

    def _perform_cosinor_analysis(self, df_window):
        y = df_window[self.sensor_col].values
        t = df_window['t_hours'].values
        omega = 2 * np.pi / self.period
        X = np.column_stack([np.ones_like(t), np.cos(omega*t), np.sin(omega*t)])
        try:
            params, residuals, rank, s = lstsq(X, y, rcond=None)
        except Exception as e:
            warnings.warn(f"Cosinor lstsq error: {e}")
            return None
        M, A_coef, B_coef = params
        amplitude = np.sqrt(A_coef**2 + B_coef**2)
        phase_radian = math.atan2(B_coef, A_coef)
        phase_hours = (phase_radian / (2 * np.pi)) * self.period
        if phase_hours < 0:
            phase_hours += self.period
        return M, amplitude, phase_hours

    def run(self):
        out = []
        for name, grp in tqdm(self.data.groupby(self.name_col), desc="Cosinor"):
            gref = grp[self.time_col].min()
            start_time = gref + pd.Timedelta(hours=self.window_length_hours)
            for ref in grp[grp[self.time_col] >= start_time][self.time_col]:
                dfw = self._prepare_window_data(grp, ref, gref)
                if dfw is None:
                    out.append({self.name_col:name, self.time_col:ref,
                                'MESOR':np.nan, 'Amplitude':np.nan, 'Phase_hours':np.nan})
                else:
                    res = self._perform_cosinor_analysis(dfw)
                    if res is None:
                        out.append({self.name_col:name, self.time_col:ref,
                                    'MESOR':np.nan, 'Amplitude':np.nan, 'Phase_hours':np.nan})
                    else:
                        M, amp, ph = res
                        out.append({self.name_col:name, self.time_col:ref,
                                    'MESOR':M, 'Amplitude':amp, 'Phase_hours':ph})
        return pd.DataFrame(out)

def PipelineSensor(df):
    df['targetTime'] = pd.to_datetime(df['targetTime'])
    enmo_48 = CosinorAnalyzer(df, 48, 0.5, 'ENMO').run()
    enmo_wk = CosinorAnalyzer(df, 168, 0.5, 'ENMO').run()
    hr_48   = CosinorAnalyzer(df, 48, 0.5, 'HR').run()
    hr_wk   = CosinorAnalyzer(df, 168, 0.5, 'HR').run()

    em = pd.merge(enmo_48, enmo_wk, on=['이름','targetTime'],
                  suffixes=('_ENMO_48h','_ENMO_week'))
    hm = pd.merge(hr_48, hr_wk, on=['이름','targetTime'],
                  suffixes=('_HR_48h','_HR_week'))
    merged = pd.merge(em, hm, on=['이름','targetTime'])
    return pd.merge(df, merged, on=['이름','targetTime'])

def create_binary_targets(df, target_col, periods):
    for p, steps in periods.items():
        df[f"{target_col}_{p}"] = (
            df.groupby('key')[target_col]
              .transform(lambda x: x.rolling(window=steps, min_periods=1).max())
              .shift(-steps).fillna(0).astype(int)
        )
    return df

def patient_split(df, group='key', train_size=0.7, random_state=42):
    """
    Return three DataFrames (train, val, test) according to hospital-specific rules.
    Hospital-specific split rules are applied based on the hospital identifier in the data.
    """
    rng = np.random.RandomState(random_state)
    hosp = df['hospital'].iat[0]

    # per-patient summary
    counts = df.groupby(group).size().reset_index(name='total_records')
    ever_pos = df.groupby(group)['dangerous_action'].max().reset_index(name='ever_positive')
    counts = counts.merge(ever_pos, on=group)
    pos_df = counts[counts['ever_positive'] == 1]
    neg_df = counts[counts['ever_positive'] == 0]

    # Hospital B: patient-level split
    if hosp == "서울대병원":
        pos_keys = pos_df[group].tolist(); rng.shuffle(pos_keys)
        assert len(pos_keys) == 14, f"SNU expects 14 positives, got {len(pos_keys)}"
        train_pos, val_pos, test_pos = pos_keys[:5], pos_keys[5:7], pos_keys[7:14]

        neg_keys = neg_df[group].tolist(); rng.shuffle(neg_keys)
        n_neg = len(neg_keys)
        n_train_neg = int(np.floor(0.7 * n_neg))
        n_val_neg   = int(np.floor(0.15 * n_neg))
        train_neg = neg_keys[:n_train_neg]
        val_neg   = neg_keys[n_train_neg:n_train_neg + n_val_neg]
        test_neg  = neg_keys[n_train_neg + n_val_neg:]

        tr = df[df[group].isin(train_pos + train_neg)].reset_index(drop=True)
        val= df[df[group].isin(val_pos   + val_neg)].reset_index(drop=True)
        te = df[df[group].isin(test_pos  + test_neg)].reset_index(drop=True)
        return tr, val, te

    # Hospital C: patient-level split
    elif hosp == "용인관리자":
        pos_keys = pos_df[group].tolist(); rng.shuffle(pos_keys)
        assert len(pos_keys) == 16, f"Expected 16 positives, got {len(pos_keys)}"
        train_pos, val_pos, test_pos = pos_keys[:8], pos_keys[10:12], pos_keys[12:16]

        neg_keys = neg_df[group].tolist(); rng.shuffle(neg_keys)
        n_neg = len(neg_keys)
        n_train_neg = int(np.floor(0.7 * n_neg))
        n_val_neg   = int(np.floor(0.15 * n_neg))
        train_neg = neg_keys[:n_train_neg]
        val_neg   = neg_keys[n_train_neg:n_train_neg + n_val_neg]
        test_neg  = neg_keys[n_train_neg + n_val_neg:]

        tr = df[df[group].isin(train_pos + train_neg)].reset_index(drop=True)
        val= df[df[group].isin(val_pos   + val_neg)].reset_index(drop=True)
        te = df[df[group].isin(test_pos  + test_neg)].reset_index(drop=True)
        return tr, val, te


    # Hospital A: patient-level split (70/15/15; ≥1 positive guaranteed in each split)
    elif hosp == "동국대":
        pos_keys = pos_df[group].tolist(); rng.shuffle(pos_keys)
        n_pos = len(pos_keys)
        n_train_pos = max(1, int(np.floor(0.70 * n_pos)))
        n_val_pos   = max(1, int(np.floor(0.15 * n_pos)))
        if n_train_pos + n_val_pos >= n_pos:   # safeguard for very few positives
            n_train_pos, n_val_pos = n_pos - 2, 1
        train_pos = pos_keys[:n_train_pos]
        val_pos   = pos_keys[n_train_pos:n_train_pos + n_val_pos]
        test_pos  = pos_keys[n_train_pos + n_val_pos:]

        neg_keys = neg_df[group].tolist(); rng.shuffle(neg_keys)
        n_neg = len(neg_keys)
        n_train_neg = int(np.floor(0.70 * n_neg))
        n_val_neg   = int(np.floor(0.15 * n_neg))
        train_neg   = neg_keys[:n_train_neg]
        val_neg     = neg_keys[n_train_neg:n_train_neg + n_val_neg]
        test_neg    = neg_keys[n_train_neg + n_val_neg:]

        tr  = df[df[group].isin(train_pos + train_neg)].reset_index(drop=True)
        val = df[df[group].isin(val_pos   + val_neg)].reset_index(drop=True)
        te  = df[df[group].isin(test_pos  + test_neg)].reset_index(drop=True)
        return tr, val, te

    # fallback: unchanged
    else:
        merged = counts.copy()
        keys_array   = merged[group].values
        labels_array = merged['ever_positive'].values
        train_size=0.3
        sss1 = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
        tr_idx, rest_idx = next(sss1.split(keys_array, labels_array))
        train_keys = keys_array[tr_idx]
        rest_keys  = keys_array[rest_idx]

        rest_labels = labels_array[rest_idx]
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=random_state)
        vi, ti = next(sss2.split(rest_keys, rest_labels))
        val_keys = rest_keys[vi]
        te_keys  = rest_keys[ti]

        tr  = df[df[group].isin(train_keys)].reset_index(drop=True)
        val = df[df[group].isin(val_keys)].reset_index(drop=True)
        te  = df[df[group].isin(te_keys)].reset_index(drop=True)
        return tr, val, te

def process_key(key, df, seq_len, feats, targs, static_feats, max_shift):
    dfk = df[df['key']==key].sort_values('targetTime')
    X, Y = dfk[feats].values, dfk[targs].values
    if len(dfk) < seq_len + max_shift:
        return []
    static = dfk[static_feats].iloc[0].values
    seqs = []
    for i in range(len(dfk) - seq_len - max_shift):
        seqs.append((static, X[i:i+seq_len], Y[i+seq_len]))
    return seqs

def gen_sequences(df, seq_len, feats, targs, static_feats, max_shift):
    seqs = []
    for k in tqdm(df['key'].unique(), desc="Keys"):
        seqs += process_key(k, df, seq_len, feats, targs, static_feats, max_shift)
    return seqs

# --- 4. MAIN LOOP (Sequence‐Creation) ---
for hosp in hospitals:
    print(f"\n--- {hosp} ---")
    dfh = data[data['hospital']==hosp].copy()

    # 4-1) patient‐level split
    tr_raw, val_raw, te_raw = patient_split(dfh)

    # 4-2) label‐encode all categorical columns
    encs = {}
    for c in categorical_columns:
        if c == 'idxInfo':
            continue
        le = LabelEncoder().fit(dfh[c].astype(str))
        encs[c] = le
        for d in (tr_raw, val_raw, te_raw):
            d[c] = le.transform(d[c].astype(str))

    if hosp == "동국대":
        # 1) split positives & negatives
        pos_df = tr_raw[tr_raw['dangerous_action'] == 1]
        neg_df = tr_raw[tr_raw['dangerous_action'] == 0]

        # 2) augment positives as usual
        augmentor_pos = DataAugmentor(sigma=0.5, num_augments=20)
        pos_aug = augmentor_pos.augment_patient_data(
            pos_df, continuous_columns, categorical_columns
        )

        # 3) augment negatives by temporarily flipping the label to reuse the augmentor
        neg_work = neg_df.copy()
        neg_work['dangerous_action'] = 1
        augmentor_neg = DataAugmentor(sigma=0.5, num_augments=20)
        neg_aug = augmentor_neg.augment_patient_data(
            neg_work, continuous_columns, categorical_columns
        )
        neg_aug['dangerous_action'] = 0

        # 4) recombine
        train_aug = pd.concat([pos_aug, neg_aug], ignore_index=True)
    elif hosp == "서울대병원":
        num_aug = 5
        augmentor = DataAugmentor(sigma=0.5, num_augments=num_aug)
        train_aug = augmentor.augment_patient_data(
            tr_raw, continuous_columns, categorical_columns
        )
    else:
        num_aug = 6
        augmentor = DataAugmentor(sigma=0.5, num_augments=num_aug)
        train_aug = augmentor.augment_patient_data(
            tr_raw, continuous_columns, categorical_columns
        )

    # 4-3) full pipeline (COSINOR + binary targets)
    def full_pipe(df):
        d2 = PipelineSensor(df.rename(columns={'key':'이름'}))
        d2.rename(columns={'이름':'key'}, inplace=True)
        # Drop features that are redundant with the raw 192-step sequence:
        #  - 48h cosinor (Amplitude/MESOR/Phase for HR & ENMO): the input window is
        #    itself 48h (192 × 15min), so cosinor fitted over the same window extracts
        #    Fourier components already visible in the raw sequence. The 168h cosinor
        #    (retained) captures circadian/weekly rhythm beyond the observation window.
        #  - 8h entropy: window too short for circadian structure; the TFT attention
        #    mechanism learns local variability directly from the sequence.
        for col in [
            'Normalized_Eight_Hour_Entropy', 'Eight_Hour_Entropy',
            'Amplitude_HR_48h', 'Amplitude_ENMO_48h',
            'MESOR_HR_48h', 'MESOR_ENMO_48h',
            'Phase_hours_HR_48h', 'Phase_hours_ENMO_48h',
        ]:
            d2.drop(columns=[col], inplace=True, errors=True)
        return create_binary_targets(d2, 'dangerous_action', PERIODS).fillna(0)

    # 4-3) full pipeline for TRAIN
    tr_proc = full_pipe(train_aug)

    # VAL (no augmentation)
    val_pos = val_raw[val_raw['dangerous_action'] == 1]
    val_neg = val_raw[val_raw['dangerous_action'] == 0]
    val_pos_aug = DataAugmentor(sigma=0.5, num_augments=0) \
        .augment_patient_data(val_pos, continuous_columns, categorical_columns)
    neg_work = val_neg.copy()
    neg_work['dangerous_action'] = 1
    val_neg_aug = DataAugmentor(sigma=0.5, num_augments=0) \
        .augment_patient_data(neg_work, continuous_columns, categorical_columns)
    val_neg_aug['dangerous_action'] = 0
    val_raw = pd.concat([val_pos_aug, val_neg_aug], ignore_index=True) \
                    .sample(frac=1, random_state=42) \
                    .reset_index(drop=True)
    val_proc = full_pipe(val_raw)

    # TEST
    te_proc = full_pipe(te_raw)

    # 4-4) prepare for sequence generation
    ALL_FEATS = [
        c for c in tr_proc.columns
        if c not in ['key','targetTime','dangerous_action','idxInfo','hospital']
           + [f"dangerous_action_{p}" for p in PERIODS]
    ]
    static_feats = [c for c in ALL_FEATS if c in categorical_columns]
    seq_feats    = [c for c in ALL_FEATS if c not in static_feats]
    targets      = [f"dangerous_action_{p}" for p in PERIODS]

    # 4-5) generate and save chunks
    for name, df_split in (('train', tr_proc), ('val', val_proc), ('test', te_proc)):
        seqs = gen_sequences(df_split, SEQ_LEN, seq_feats, targets, static_feats, MAX_SHIFT)
        if not seqs:
            print(f"  → SKIPPING {hosp}_{name}: no sequences generated")
            continue
        nchunks = math.ceil(len(seqs) / CHUNK_SIZE)
        for i in range(nchunks):
            chunk = seqs[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]
            fname = f"{hosp}_{name}_chunk_{i}.pkl"
            with open(os.path.join(OUT_DIR, fname), 'wb') as f:
                pickle.dump(chunk, f)
            print(f"  → {fname} ({len(chunk)} seqs)")

---
## Stage 2: Memmap Creation
Reads pickles from `processed_sequences_by_hospital/` → writes `.npy` memmaps to `last_npy_data/`

In [ ]:
import os
import math
import pickle
import gc
import logging
from collections import defaultdict

import numpy as np
from sklearn.model_selection import train_test_split

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
SEQS_DIR        = "patient_level_split/processed_sequences_by_hospital"
ROOT_DIR        = "patient_level_split/last_npy_data"
GLOBAL_DIR      = os.path.join(ROOT_DIR, "GlobalData")
HOSP_DIR        = os.path.join(ROOT_DIR, "HospitalsData")
CHUNK_SIZE      = 100   # controls how many samples we read/write at once

os.makedirs(GLOBAL_DIR, exist_ok=True)
os.makedirs(HOSP_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# LOGGING SETUP
# ----------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

# ----------------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------------
def filter_static_features(static_x):
    arr = np.array(static_x, dtype=object)
    filtered = [v for v in arr if not isinstance(v, str)]
    if not filtered:
        return None
    return np.array(filtered, dtype=np.float32)

def remove_non_numeric_from_sequence(seq_x):
    if not hasattr(seq_x, "__len__") or len(seq_x) == 0:
        return seq_x
    first = seq_x[0]
    non_numeric = {j for j,v in enumerate(first) if isinstance(v, str)}
    if not non_numeric:
        return seq_x
    cleaned = []
    for row in seq_x:
        cleaned.append([v for j,v in enumerate(row) if j not in non_numeric])
    return cleaned

def write_memmap_chunked(path, total_n, sample_shape, dtype=np.float32):
    mm = np.memmap(path, mode='w+', dtype=dtype, shape=(total_n,) + sample_shape)
    return mm, 0

def append_to_memmap(mm_obj, write_idx, batch_array):
    n = batch_array.shape[0]
    mm_obj[write_idx:write_idx+n] = batch_array
    return write_idx + n

# ----------------------------------------------------------------------
# 1) Group all chunk files by hospital & split
# ----------------------------------------------------------------------
hospital_files = defaultdict(lambda: {"train": [], "val": [], "test": []})
for fn in os.listdir(SEQS_DIR):
    if not fn.endswith(".pkl") or "_chunk_" not in fn:
        continue
    head = fn.split("_chunk_")[0]
    if head.endswith("_train"):
        split = "train"; hosp = head[:-len("_train")]
    elif head.endswith("_val"):
        split = "val";   hosp = head[:-len("_val")]
    elif head.endswith("_test"):
        split = "test";  hosp = head[:-len("_test")]
    else:
        continue
    hospital_files[hosp][split].append(os.path.join(SEQS_DIR, fn))

for hosp, splits in hospital_files.items():
    for split in ("train","val","test"):
        splits[split].sort(key=lambda p: int(os.path.basename(p).split("_chunk_")[-1].split(".")[0]))

# ----------------------------------------------------------------------
# 2) MAIN LOOP: For each hospital, build counts and assign all test → global & local
# ----------------------------------------------------------------------
global_total_count = 0
hosp_info = {}

for hosp, splits in hospital_files.items():
    logging.info(f"=== Counting samples for hospital: {hosp} ===")
    info = {}
    # TRAIN & VAL counts
    for split in ("train","val"):
        total_valid = 0
        for pkl_path in splits[split]:
            with open(pkl_path, "rb") as f:
                seq_list = pickle.load(f)
            for (s,x,y) in seq_list:
                fs = filter_static_features(s)
                if fs is None:
                    continue
                rx = remove_non_numeric_from_sequence(x)
                try:
                    _ = np.asarray(rx, dtype=np.float32)
                    _ = np.asarray(y, dtype=np.float32)
                except:
                    continue
                total_valid += 1
            del seq_list; gc.collect()
        info[f"{split}_count"] = total_valid
        logging.info(f"  {split.upper()}: {total_valid} valid samples")

    # TEST: count and assign all to global & local
    test_labels = []
    for pkl_path in splits["test"]:
        with open(pkl_path, "rb") as f:
            seq_list = pickle.load(f)
        for (s,x,y) in seq_list:
            if filter_static_features(s) is None:
                continue
            test_labels.append(1 if np.asarray(y).sum()>0 else 0)
        del seq_list; gc.collect()

    N_test_valid = len(test_labels)
    info["test_count"] = N_test_valid
    logging.info(f"  TEST: {N_test_valid} valid samples (all → GLOBAL & LOCAL)")

    all_idx = list(range(N_test_valid))
    info["test_global_idx"] = set(all_idx)
    info["test_local_idx"]  = set(all_idx)
    info["test_global_count"] = N_test_valid
    info["test_local_count"]  = N_test_valid
    global_total_count += N_test_valid

    logging.info(f"    → {N_test_valid} → GLOBAL, {N_test_valid} → LOCAL")
    hosp_info[hosp] = info

# ----------------------------------------------------------------------
# 3) Create empty memmaps for global & per-hospital
# ----------------------------------------------------------------------
# Determine sample shapes from one example
some_hosp = next(iter(hospital_files))
some_files = hospital_files[some_hosp]["train"] or hospital_files[some_hosp]["val"] or hospital_files[some_hosp]["test"]
with open(some_files[0], "rb") as f:
    sample_list = pickle.load(f)
s0,x0,y0 = sample_list[0]
fs0 = filter_static_features(s0)
rx0 = remove_non_numeric_from_sequence(x0)
static_dim = fs0.shape[0]
seq_shape  = np.asarray(rx0, dtype=np.float32).shape
targs_dim  = np.asarray(y0, dtype=np.float32).shape[0]
del sample_list, fs0, rx0, y0; gc.collect()

# GLOBAL memmaps
global_static_mm, g_s_ptr = write_memmap_chunked(
    os.path.join(GLOBAL_DIR, "static_data.npy"),
    total_n=global_total_count,
    sample_shape=(static_dim,)
)
global_seq_mm,   g_x_ptr = write_memmap_chunked(
    os.path.join(GLOBAL_DIR, "sequence_data.npy"),
    total_n=global_total_count,
    sample_shape=seq_shape
)
global_targs_mm, g_y_ptr = write_memmap_chunked(
    os.path.join(GLOBAL_DIR, "targets.npy"),
    total_n=global_total_count,
    sample_shape=(targs_dim,)
)

# per‐hospital memmaps
for hosp, info in hosp_info.items():
    out_dir = os.path.join(HOSP_DIR, hosp)
    os.makedirs(out_dir, exist_ok=True)

    # TRAIN
    n_tr = info["train_count"]
    if n_tr > 0:
        tr_s_mm, tr_s_ptr = write_memmap_chunked(
            os.path.join(out_dir, "static_train.npy"),
            total_n=n_tr,
            sample_shape=(static_dim,)
        )
        tr_x_mm, tr_x_ptr = write_memmap_chunked(
            os.path.join(out_dir, "sequence_train.npy"),
            total_n=n_tr,
            sample_shape=seq_shape
        )
        tr_y_mm, tr_y_ptr = write_memmap_chunked(
            os.path.join(out_dir, "targets_train.npy"),
            total_n=n_tr,
            sample_shape=(targs_dim,)
        )
    else:
        tr_s_mm = tr_x_mm = tr_y_mm = None
        tr_s_ptr = tr_x_ptr = tr_y_ptr = 0

    # VAL
    n_val = info["val_count"]
    if n_val > 0:
        v_s_mm, v_s_ptr = write_memmap_chunked(
            os.path.join(out_dir, "static_val.npy"),
            total_n=n_val,
            sample_shape=(static_dim,)
        )
        v_x_mm, v_x_ptr = write_memmap_chunked(
            os.path.join(out_dir, "sequence_val.npy"),
            total_n=n_val,
            sample_shape=seq_shape
        )
        v_y_mm, v_y_ptr = write_memmap_chunked(
            os.path.join(out_dir, "targets_val.npy"),
            total_n=n_val,
            sample_shape=(targs_dim,)
        )
    else:
        v_s_mm = v_x_mm = v_y_mm = None
        v_s_ptr = v_x_ptr = v_y_ptr = 0

    # LOCAL TEST
    n_loc = info["test_local_count"]
    if n_loc > 0:
        t_s_mm, t_s_ptr = write_memmap_chunked(
            os.path.join(out_dir, "static_test.npy"),
            total_n=n_loc,
            sample_shape=(static_dim,)
        )
        t_x_mm, t_x_ptr = write_memmap_chunked(
            os.path.join(out_dir, "sequence_test.npy"),
            total_n=n_loc,
            sample_shape=seq_shape
        )
        t_y_mm, t_y_ptr = write_memmap_chunked(
            os.path.join(out_dir, "targets_test.npy"),
            total_n=n_loc,
            sample_shape=(targs_dim,)
        )
    else:
        t_s_mm = t_x_mm = t_y_mm = None
        t_s_ptr = t_x_ptr = t_y_ptr = 0

    info["train_ptrs"] = (tr_s_mm, tr_x_mm, tr_y_mm, tr_s_ptr, tr_x_ptr, tr_y_ptr)
    info["val_ptrs"]   = (v_s_mm, v_x_mm, v_y_mm,   v_s_ptr, v_x_ptr, v_y_ptr)
    info["test_ptrs"]  = (t_s_mm, t_x_mm, t_y_mm,   t_s_ptr, t_x_ptr, t_y_ptr)

logging.info("Memmaps created; now populating...")

# ----------------------------------------------------------------------
# 4) Populate memmaps chunk by chunk
# ----------------------------------------------------------------------
for hosp, splits in hospital_files.items():
    info = hosp_info[hosp]
    (tr_s_mm, tr_x_mm, tr_y_mm, tr_s_ptr, tr_x_ptr, tr_y_ptr) = info["train_ptrs"]
    (v_s_mm, v_x_mm, v_y_mm,   v_s_ptr, v_x_ptr, v_y_ptr)   = info["val_ptrs"]
    (t_s_mm, t_x_mm, t_y_mm,   t_s_ptr, t_x_ptr, t_y_ptr)   = info["test_ptrs"]

    # TRAIN
    for pkl in splits["train"]:
        with open(pkl, "rb") as f:
            seqs = pickle.load(f)
        buf_s, buf_x, buf_y = [], [], []
        for (s, x, y) in seqs:
            fs = filter_static_features(s)
            if fs is None: continue
            rx = remove_non_numeric_from_sequence(x)
            try:
                ax = np.array(rx, dtype=np.float32)
                ay = np.array(y, dtype=np.float32)
            except:
                continue
            buf_s.append(fs); buf_x.append(ax); buf_y.append(ay)
            if len(buf_s) >= CHUNK_SIZE:
                batch_s = np.stack(buf_s)
                batch_x = np.stack(buf_x)
                batch_y = np.stack(buf_y)
                tr_s_ptr = append_to_memmap(tr_s_mm, tr_s_ptr, batch_s)
                tr_x_ptr = append_to_memmap(tr_x_mm, tr_x_ptr, batch_x)
                tr_y_ptr = append_to_memmap(tr_y_mm, tr_y_ptr, batch_y)
                buf_s.clear(); buf_x.clear(); buf_y.clear(); gc.collect()
        if buf_s:
            batch_s = np.stack(buf_s)
            batch_x = np.stack(buf_x)
            batch_y = np.stack(buf_y)
            tr_s_ptr = append_to_memmap(tr_s_mm, tr_s_ptr, batch_s)
            tr_x_ptr = append_to_memmap(tr_x_mm, tr_x_ptr, batch_x)
            tr_y_ptr = append_to_memmap(tr_y_mm, tr_y_ptr, batch_y)
            buf_s.clear(); buf_x.clear(); buf_y.clear(); gc.collect()
        del seqs; gc.collect()
    if tr_s_mm is not None:
        tr_s_mm.flush(); tr_x_mm.flush(); tr_y_mm.flush()
    logging.info(f"  → TRAIN done for {hosp}: {tr_s_ptr} samples")

    # VAL
    for pkl in splits["val"]:
        with open(pkl, "rb") as f:
            seqs = pickle.load(f)
        buf_s, buf_x, buf_y = [], [], []
        for (s, x, y) in seqs:
            fs = filter_static_features(s)
            if fs is None: continue
            rx = remove_non_numeric_from_sequence(x)
            try:
                ax = np.array(rx, dtype=np.float32)
                ay = np.array(y, dtype=np.float32)
            except:
                continue
            buf_s.append(fs); buf_x.append(ax); buf_y.append(ay)
            if len(buf_s) >= CHUNK_SIZE:
                batch_s = np.stack(buf_s)
                batch_x = np.stack(buf_x)
                batch_y = np.stack(buf_y)
                v_s_ptr = append_to_memmap(v_s_mm, v_s_ptr, batch_s)
                v_x_ptr = append_to_memmap(v_x_mm, v_x_ptr, batch_x)
                v_y_ptr = append_to_memmap(v_y_mm, v_y_ptr, batch_y)
                buf_s.clear(); buf_x.clear(); buf_y.clear(); gc.collect()
        if buf_s:
            batch_s = np.stack(buf_s)
            batch_x = np.stack(buf_x)
            batch_y = np.stack(buf_y)
            v_s_ptr = append_to_memmap(v_s_mm, v_s_ptr, batch_s)
            v_x_ptr = append_to_memmap(v_x_mm, v_x_ptr, batch_x)
            v_y_ptr = append_to_memmap(v_y_mm, v_y_ptr, batch_y)
            buf_s.clear(); buf_x.clear(); buf_y.clear(); gc.collect()
        del seqs; gc.collect()
    if v_s_mm is not None:
        v_s_mm.flush(); v_x_mm.flush(); v_y_mm.flush()
    logging.info(f"  → VAL done  for {hosp}: {v_s_ptr} samples")

    # TEST → GLOBAL & LOCAL
    cur = 0
    for pkl in splits["test"]:
        with open(pkl, "rb") as f:
            seqs = pickle.load(f)
        for (s, x, y) in seqs:
            fs = filter_static_features(s)
            if fs is None:
                cur += 1
                continue
            rx = remove_non_numeric_from_sequence(x)
            try:
                ax = np.array(rx, dtype=np.float32)
                ay = np.array(y, dtype=np.float32)
            except:
                cur += 1
                continue
            # ALWAYS write to GLOBAL
            g_s_ptr = append_to_memmap(global_static_mm, g_s_ptr, fs[None, :])
            g_x_ptr = append_to_memmap(global_seq_mm,   g_x_ptr, ax[None, :, :])
            g_y_ptr = append_to_memmap(global_targs_mm, g_y_ptr, ay[None, :])
            # ALWAYS write to LOCAL TEST
            t_s_ptr = append_to_memmap(t_s_mm, t_s_ptr, fs[None, :])
            t_x_ptr = append_to_memmap(t_x_mm, t_x_ptr, ax[None, :, :])
            t_y_ptr = append_to_memmap(t_y_mm, t_y_ptr, ay[None, :])
            cur += 1
        del seqs; gc.collect()
    if t_s_mm is not None:
        t_s_mm.flush(); t_x_mm.flush(); t_y_mm.flush()
    logging.info(f"  → TEST written locally for {hosp}: {info['test_local_count']} samples")

# ----------------------------------------------------------------------
# 5) Flush GLOBAL memmaps at end
# ----------------------------------------------------------------------
global_static_mm.flush()
global_seq_mm.flush()
global_targs_mm.flush()
logging.info(f"✅ GLOBAL done: {g_s_ptr} samples")
print("Done.")
